In [1]:
# Importações básicas
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')

# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df,
    parse_utm_string,
    timestamp_ms_to_datetime
)

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc, load_query_from_file

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart

# utilitários tmb
from tmb_utils import TMB  

# Utiliários Tally Forms
from tally_utils import Tally

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()

# Inicializar API TMB
tmb = TMB()

# Inicializar API Tally
tally = Tally()

print('✓ Importações concluídas com sucesso!')

✓ Importações concluídas com sucesso!


# Load Dataframes

In [2]:
df_ofan = db.execute_query_from_file('select_complete_lead_data_from_campaign', params={'conversion_type_id': '1', 'campaign_id': 'OFAN_25'})
df_ofan['phone'] = process_phone_string(df_ofan, 'phone')

print('df_ofan shape:', df_ofan.shape)

df_ofan_r = db.execute_query_from_file('select_complete_lead_data_from_campaign', params={'conversion_type_id': '1', 'campaign_id': 'OFAN_25R'})
df_ofan_r['phone'] = process_phone_string(df_ofan_r, 'phone')

print('df_ofan_r shape:', df_ofan_r.shape)

df_ofan_met = db.execute_query_from_file('select_complete_lead_data_from_campaign', params={'conversion_type_id': '1', 'campaign_id': 'OFAN_25RMET'})
df_ofan_met['phone'] = process_phone_string(df_ofan_met, 'phone')

print('df_ofan_met shape:', df_ofan_met.shape)

df_ofan shape: (23930, 18)
df_ofan_r shape: (12787, 18)
df_ofan_met shape: (1375, 18)


# Buyers Check

In [ ]:
df_hotmart_pt_1 =  load_raw_data('hotmart_pt_1.csv', data_dir='../data/interim')

df_hotmart_pt_2 = load_raw_data('hotmart_pt_2.csv', data_dir='../data/interim')

df_hotmart_pt_3 = flatten_list_to_df(hotmart.get_sales_history(
    start_date='2025-12-01',
    end_date='2025-12-31',
))

df_hotmart = pd.concat([df_hotmart_pt_1, df_hotmart_pt_2, df_hotmart_pt_3])

✓ Concluído: 16 página(s) processada(s). Total de 783 itens obtidos.)


In [4]:
ofan_offers = db.execute_query_from_file('select_offers_where_campaign_like', params={'campaign_id_like': '%ofan%'})
lead_ids = db.execute_query_from_file('select_lead_id_by_purchase')

In [5]:
df_sales = df_hotmart.copy()

In [6]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(df_sales):
    df_sales = pd.merge(df_hotmart, ofan_offers, left_on='purchase_offer_code', right_on='offer_id', how='left')
    df_sales = pd.merge(df_sales, lead_ids, left_on='purchase_transaction', right_on='transaction', how='left')

    #df_sales['campaign_id_'] = df_sales['campaign_id']

    # Filter rows based on column: 'offer_id'
    df_sales = df_sales[df_sales['offer_id'].notna()]
    # Filter rows based on column: 'purchase_recurrency_number'
    df_sales = df_sales[(df_sales['purchase_recurrency_number'] == 1) | (df_sales['purchase_recurrency_number'].isna())]

    # Drop columns: 'purchase_warranty_expire_date', 'purchase_tracking_source' and 22 other columns
    df_sales = df_sales.drop(columns=['purchase_warranty_expire_date', 'purchase_tracking_source', 'producer_ucode', 'producer_name', 'purchase_tracking_external_code', 'offer_id', 'updated_at', 'created_at', 'status', 'name', 'description', 'start_date', 'end_date', 'price', 'main_product_id', 'access_duration_months', 'is_default', 'entry_value', 'installment_12x_value', 'platform', 'base_url', 'is_child', 'comercial_use', 'transaction'])
    # Drop duplicate rows in column: 'purchase_transaction'
    df_sales = df_sales.drop_duplicates(subset=['purchase_transaction'])
    # Filter rows based on column: 'campaign_id'
    df_sales = df_sales[df_sales['campaign_id'].notna()]
    
    # Parse UTM string DEPOIS de todos os merges e filtros
    #df_sales = parse_utm_string(df_sales, 'purchase_tracking_source_sck')
    
    return df_sales


df_sales = clean_data(df_sales.copy())
df_sales = df_sales.reset_index(drop=True)

In [7]:
df_cdf_sales = df_sales[df_sales['product_name'] == 'Cronograma dos Fluentes ']

In [8]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(df):
    df = pd.merge(df, df_cdf_sales, on='lead_id', how='left')
    # Drop columns: 'purchase_status', 'purchase_hotmart_fee_currency_code' and 21 other columns
    df = df.drop(columns=['purchase_status', 'purchase_hotmart_fee_currency_code', 'purchase_hotmart_fee_total', 'purchase_hotmart_fee_percentage', 'purchase_hotmart_fee_fixed', 'purchase_hotmart_fee_base', 'purchase_commission_as', 'purchase_is_subscription', 'purchase_payment_installments_number', 'purchase_payment_type', 'purchase_order_date', 'purchase_offer_payment_mode', 'buyer_name', 'buyer_ucode', 'buyer_email', 'purchase_recurrency_number'])
    return df

df_ofan = clean_data(df_ofan.copy())
df_ofan_r = clean_data(df_ofan_r.copy())
df_ofan_met = clean_data(df_ofan_met.copy())


In [9]:
df_ofan.to_csv('df_ofan.csv', index=False)
df_ofan_r.to_csv('df_ofan_r.csv', index=False)
df_ofan_met.to_csv('df_ofan_met.csv', index=False)